In [127]:
import os
import numpy as np
import pandas as pd
from math import *

In [60]:
RATING_PATH = './ml-latest-small/ratings.csv'
MOVIES_PATH = './ml-latest-small/movies.csv'
CACHE_DIR = './cache'

def load_data(ratings_file, movies_file):
    cache_path = os.path.join(CACHE_DIR, 'ratings.cache')
    if os.path.exists(cache_path):  # 判断是否存在缓存文件
        print('from cache...')
        ratings = pd.read_pickle(cache_path)
    else:
        dtype = {'userId': np.int32, 'movieId': np.int32, 'rating': np.float32}
        ratings = pd.read_csv(ratings_file, dtype=dtype, usecols=range(3))

        dtype = {'movieId': np.int32, 'title': str, 'genres': str}
        movies = pd.read_csv(movies_file, dtype=dtype, usecols=range(3))
        movies['movieIdx'] = movies.index + 1
        movies = movies[['movieId', 'movieIdx']]

        ratings = pd.merge(ratings, movies, on='movieId')
        ratings.to_pickle(cache_path)
    
    return ratings

In [147]:
class BaselineCF(object):
    def __init__(self, top, dataset, columns=('user_id', 'item_id', 'rating')):
        self.top = top
        self.dataset = dataset
        self.columns = columns

    def fit(self):
        # user rating
        self.user_ratings = dataset.groupby(self.columns[0]).agg([list])[[self.columns[3], self.columns[2]]]
        # movie rating
        self.movie_ratings = dataset.groupby(self.columns[3]).agg([list])[[self.columns[0], self.columns[2]]]
        
    def getNearestNeighbor(self, userId):

        self.neighbors = []
        
        neighbors = []
        for mid in self.user_ratings.loc[userId, 'movieIdx'].values[0]:
            for uid in self.movie_ratings.loc[mid , 'userId'].values[0]: # todo 根据行号取电影id
                if(uid != userId and uid not in neighbors):
                    neighbors.append(uid)
                        
        # print(neighbors)
        for uid in neighbors:
            score = self.getSimScore(userId, uid)
            self.neighbors.append([score, uid])
            
            self.neighbors.sort(reverse=True)
            self.neighbors = self.neighbors[:self.top]
        
    def formatuserDict(self, userId, l):
        user = {}
        me = self.user_ratings.loc[userId, 'movieIdx'].values[0]
        meRating = self.user_ratings.loc[userId, 'rating'].values[0]
        
        other = self.user_ratings.loc[l, 'movieIdx'].values[0]
        otherRting = self.user_ratings.loc[l, 'rating'].values[0]
        
        for i in range(len(me)):
            mIdx, rating = me[i], meRating[i] / 5
            user[mIdx] = [rating, 0]
        
        for i in range(len(other)):
            mIdx, rating = other[i], otherRting[i] / 5
            if mIdx not in user:
                user[mIdx] = [0, rating]
            else:
                user[mIdx][1] = rating
        
        return user
       
    def getSimScore(self, userId, uid):
        user = self.formatuserDict(userId, uid)
        x = 0.0
        y = 0.0
        z = 0.0
        
        for k, v in user.items():
            x += float(v[0]) * float(v[0])
            y += float(v[1]) * float(v[1])
            z += float(v[0]) * float(v[1])
        
        if(z == 0.0):
            return 0
        return z / sqrt(x * y)
    
    
    def getrecommandList(self, userId):
        
        self.getNearestNeighbor(userId)
        
        self.recommandList = []
        recommandDict = {}
    
        for neighbor in self.neighbors:
            score, simUser = neighbor
            movies = self.user_ratings.loc[simUser, 'movieIdx'].values[0]
            rating = self.user_ratings.loc[simUser, 'rating'].values[0]
            
            for i in range(len(movies)):
                m, r = movies[i], rating[i]
                if(m in recommandDict):
                    recommandDict[m] += r * score
                else:
                    recommandDict[m] = r * score

        for key in recommandDict:
            self.recommandList.append([recommandDict[key], key])
        
        self.recommandList.sort(reverse=True)
        self.recommandList = self.recommandList[:50]

In [148]:
dataset = load_data(RATING_PATH, MOVIES_PATH)
dataset.columns
columns = ('userId', 'movieId', 'rating', 'movieIdx')
cf = BaselineCF(10, dataset, columns)
cf.fit()

from cache...


In [150]:
cf.getrecommandList(5)

In [151]:
for i in cf.recommandList:
    print(i)

[20.52155523905946, 124]
[19.75507078777442, 315]
[19.69670395250512, 399]
[19.59600075562896, 509]
[19.239304950354335, 258]
[18.317314475896186, 338]
[18.23658263545259, 278]
[18.022761112617186, 462]
[16.971296764662544, 335]
[15.893915887520329, 261]
[15.887109424343597, 508]
[15.796512173655557, 98]
[15.378677394060375, 323]
[14.96005646076012, 473]
[14.852282246997968, 419]
[14.726253902928368, 507]
[14.714892959743555, 33]
[14.572515114844467, 513]
[14.278034180196947, 32]
[13.990532865814814, 515]
[13.875957155113193, 308]
[13.750578179319476, 139]
[13.622504428506575, 44]
[13.593174947058849, 135]
[12.949346457412869, 510]
[12.905207703810902, 413]
[11.945653565965703, 396]
[11.615570662559136, 437]
[11.594382687451908, 127]
[11.474907844621264, 379]
[11.398744372106279, 316]
[11.163063386400305, 303]
[11.138952007140915, 326]
[11.017809194897229, 47]
[10.947327691815932, 506]
[10.856810647992033, 157]
[10.790827953146067, 254]
[10.03221566608409, 193]
[9.910368208570421, 21]


In [71]:
cf.user_ratings

,movieIdx,rating
,list,list
userId,,
1,"[1, 3, 6, 44, 47, 63, 90, 98, 125, 131, 137, 1...","[4.0, 4.0, 4.0, 5.0, 5.0, 3.0, 5.0, 4.0, 5.0, ..."
2,"[292, 2675, 278, 1285, 4616, 5306, 6254, 6316,...","[4.0, 4.0, 3.0, 4.5, 4.0, 3.5, 4.0, 4.0, 4.5, ..."
3,"[462, 975, 1191, 1494, 1554, 1568, 2766, 31, 5...","[0.5, 3.5, 4.5, 0.5, 0.5, 2.0, 5.0, 0.5, 0.5, ..."
4,"[44, 202, 225, 258, 385, 399, 486, 511, 521, 5...","[2.0, 2.0, 5.0, 1.0, 1.0, 5.0, 2.0, 5.0, 5.0, ..."
5,"[1, 47, 98, 258, 276, 308, 326, 399, 462, 509,...","[4.0, 4.0, 4.0, 5.0, 2.0, 3.0, 4.0, 4.0, 5.0, ..."
...,...,...
606,"[1, 44, 47, 63, 98, 131, 191, 198, 202, 225, 2...","[2.5, 3.0, 4.5, 4.0, 3.5, 4.0, 4.5, 2.5, 3.5, ..."
607,"[1, 98, 225, 258, 276, 368, 399, 419, 462, 486...","[4.0, 5.0, 3.0, 3.0, 3.0, 3.0, 5.0, 4.0, 5.0, ..."


In [65]:
cf.movie_ratings

,userId,rating
,list,list
movieIdx,,
1,"[1, 5, 7, 15, 17, 18, 19, 21, 27, 31, 32, 33, ...","[4.0, 4.0, 4.5, 2.5, 4.5, 3.5, 4.0, 3.5, 3.0, ..."
2,"[6, 8, 18, 19, 20, 21, 27, 51, 62, 68, 82, 91,...","[4.0, 4.0, 3.0, 3.0, 3.0, 3.5, 4.0, 4.5, 4.0, ..."
3,"[1, 6, 19, 32, 42, 43, 44, 51, 58, 64, 68, 91,...","[4.0, 5.0, 3.0, 3.0, 4.0, 5.0, 3.0, 4.0, 3.0, ..."
4,"[6, 14, 84, 162, 262, 411, 600]","[3.0, 3.0, 3.0, 3.0, 1.0, 2.0, 1.5]"
5,"[6, 31, 43, 45, 58, 66, 68, 84, 103, 107, 111,...","[5.0, 3.0, 5.0, 3.0, 4.0, 4.0, 2.0, 3.0, 4.0, ..."
...,...,...
9738,[184],[4.0]
9739,[184],[3.5]
